**`05_ingest_global_administrative_units`**

Import geometries from the Global Administrative Database (GADM), and
build the global ISO+GADM reference spine, `admin-spine-2026`.

You will not normally need to run this: the spine's `admin_id`
reference table already ships with `openplaces`, and
`02_set_up_admin_scope.ipynb` is the notebook to run for your own
project setup. This one exists for two reasons:

- it documents how `admin-spine-2026` was originally built, in case
  it ever needs to be regenerated (e.g. after a GADM update), and
- it is still the way to download GADM's actual boundary geometries
  (>2 GB) locally, if you want to make maps or do geoprocessing with
  them -- those are not shipped, only the lightweight ID/name
  reference table is.

# Administrative units identifiers
``openplaces`` organizes its data by ``admin_id`` (data class: ``AdminId``).

``admin_id`` is a geographical administrative index with hierarchical ``.levels`` of any depth.

- **1** - countries
- **2** - states/departments/...
- **3** - counties/municipalities/...
- **4** - subdivisions/towns/...

In [ ]:
import argparse

from openplaces.api import get_admin
from openplaces.io.admin import get_admin1_iso, get_admin2_iso
from openplaces.io.ingester import Ingester
from openplaces.path import recipe_path
from openplaces.recipe import get_recipe_by_id
from openplaces.utils import pretty_print

In [ ]:
# Define arguments
parser = argparse.ArgumentParser(description='Ingest admin units using a recipe')

parser.add_argument(
    '--reprocess',
    help='Reprocess input data from downloaded file',
    action='store_true',
)
parser.add_argument(
    '--redownload',
    help='Redownload input data from original source',
    action='store_true',
)
parser.add_argument(
    '--create_admin_spine',
    help='Update the reference set of administrative units',
    action='store_true',
)
parser.add_argument(
    '--keep_unzipped',
    help='If True, keeps unzipped files instead of deleting them',
    action='store_true',
)
parser.add_argument(
    '--verbose',
    help='If T',
    action='store_true',
)

# Test arguments

In [ ]:
ARGS_TEST = (
    '--reprocess '
    # '--redownload '
    # '--create_admin_spine '
    # '--keep_unzipped'
    '--verbose '
)

# Convert argument string to list of strings
args_list = [x for x in ARGS_TEST.split(' ') if x != '']

# Parse list of arguments
args = parser.parse_args(args_list)

# Display arguments to check if parsing worked as expected
args

In [ ]:
ADMIN_SPINE = 'admin-openplaces-2026'

ADMIN_RECIPE_IDS = {
    1: 'admin-gadm-4~1_admin1',
    2: 'admin-gadm-4~1_admin2',
    3: 'admin-gadm-4~1_admin3',
    4: 'admin-gadm-4~1_admin4',
}

# `admin1`: countries / territories
The highest level of the administrative hierarchy.
## ISO countries
Top-level administrative identifiers, gap-filled to match GADM, ships with `openplaces`

In [ ]:
get_admin1_iso().sample(5).sort_index()

## GADM level 0
GADM starts at 0 = countries (openplaces 1 = countries)

In [ ]:
pretty_print(get_recipe_by_id(ADMIN_RECIPE_IDS[1]))

In [ ]:
ingester = Ingester(ADMIN_RECIPE_IDS[1], verbose=args.verbose)
ingester.ingest(
    reprocess=args.reprocess, redownload=args.redownload, keep_unzipped=True
)

In [ ]:
# Save as initial `openplaces` Admin-1 recipe
if args.create_admin_spine:
    openplaces_recipe_path = recipe_path(None, ADMIN_SPINE, filename='admin1.csv')
    openplaces_recipe_path.parent.mkdir(parents=True, exist_ok=True)
    get_admin(level=1, recipe=ADMIN_RECIPE_IDS[1], all_columns=True).to_csv(
        openplaces_recipe_path, encoding='utf-8-sig'
    )

In [ ]:
# Read result (GADM is default geometry)
admin1 = get_admin(level=1, geom=True)
admin1.sample(5).sort_index()

# ``admin2``: states / departments

## ISO states / departments

In [ ]:
admin2_iso = get_admin2_iso()
admin2_iso.sample(5).sort_index()

## GADM states / departments

In [ ]:
pretty_print(get_recipe_by_id(ADMIN_RECIPE_IDS[2]))

In [ ]:
ingester = Ingester(ADMIN_RECIPE_IDS[2], verbose=args.verbose)
ingester.ingest(reprocess=args.reprocess, keep_unzipped=True)

In [ ]:
# Save as initial `openplaces` Admin-2 recipe
if args.create_admin_spine:
    ADMIN2_COLUMNS = [
        'name',
        'type',
        'name_original',
        'name_alternatives',
        'type_orginal',
        'admin2_id_gadm',
        'admin2_id_admin1',
        'admin2_id_source',
    ]
    get_admin(level=2, recipe=ADMIN_RECIPE_IDS[2], columns=ADMIN2_COLUMNS).replace(
        'NA', ''
    ).to_csv(
        recipe_path(None, ADMIN_SPINE, filename='admin2.csv'),
        encoding='utf-8-sig',
    )

In [ ]:
admin2 = get_admin(level=2)
admin2.sample(5).sort_index()

# ``admin3``: counties / municipalities

## GADM counties / municipalities

In [ ]:
pretty_print(get_recipe_by_id(ADMIN_RECIPE_IDS[3]))

In [ ]:
ingester = Ingester(ADMIN_RECIPE_IDS[3], verbose=args.verbose)
ingester.ingest(reprocess=args.reprocess, keep_unzipped=True)

In [ ]:
# Save as initial `openplaces` Admin-3 recipe
if args.create_admin_spine:
    ADMIN3_COLUMNS = [
        'name',
        'type',
        'name_original',
        'name_alternatives',
        'type_orginal',
        'admin3_id_gadm',
        'admin3_id_admin1',
        'admin3_id_source',
    ]

    get_admin(level=3, recipe=ADMIN_RECIPE_IDS[3], columns=ADMIN3_COLUMNS).replace(
        'NA', ''
    ).to_csv(
        recipe_path(None, ADMIN_SPINE, filename='admin3.csv'),
        encoding='utf-8-sig',
    )

In [ ]:
admin3 = get_admin(level=3)
admin3.sample(5).sort_index()

# ``admin4``: towns / county subdivisions / 

## GADM towns / county subdivisions / ...

In [ ]:
pretty_print(get_recipe_by_id(ADMIN_RECIPE_IDS[4]))

In [ ]:
ingester = Ingester(ADMIN_RECIPE_IDS[4], verbose=args.verbose)
ingester.ingest(reprocess=args.reprocess, keep_unzipped=args.keep_unzipped)

In [ ]:
# Save as initial `openplaces` Admin-4 recipe
if args.create_admin_spine:
    ADMIN4_COLUMNS = [
        'name',
        'type',
        'name_original',
        'name_alternatives',
        'type_original',
        'admin4_id_admin1',
        'admin4_id_gadm',
        'admin4_id_source',
    ]
    get_admin(level=4, recipe=ADMIN_RECIPE_IDS[4], columns=ADMIN4_COLUMNS).replace(
        'NA', ''
    ).to_csv(
        recipe_path(None, ADMIN_SPINE, filename='admin4.csv'),
        encoding='utf-8-sig',
    )

In [ ]:
admin4 = get_admin(level=4)
admin4.sample(5).sort_index()